## Simple vacancy calculation for InAuSe2

This notebook can be used as a template for running other defect calculations for other Chalcopyrite as well as other materials.

In [1]:
%load_ext aiida
%aiida

Loaded AiiDA DB environment - profile name: bit.

NOTE: this notebook requires the latest version of aiida-grouppathx.
You can update it with:
```
pip install -U git+https://github.com/zhubonan/aiida-grouppathx
```

In [2]:
from aiida_vasp.workchains.v2 import VaspRelaxUpdater, VaspBuilderUpdater
from aiida_grouppathx import GroupPathX, decorate_with_exit_status
from ase.io import read
from ase.visualize import view
from aiida import orm
from aiida_user_addons.process.transform import make_vac, make_supercell, rattle, get_primitive
import time

In [3]:
from aiida_user_addons.tools.pymatgen import load_mp_struct

In [4]:
FORMULA='InAuSe2' # <- Define the chemical formula of the material here

In [5]:
basepath = GroupPathX(f'{FORMULA.lower()}-defect')
workpath = basepath['workflows']


## Compute elemental energies

If these calculations has been done already - we simply need to import them into the current GroupPath.
Otherwise, we do calculation for the elemental phases and add them to the group

In [6]:
structure = load_mp_struct('mp-81')
elem = structure.get_ase().get_chemical_symbols()[0]
print(elem)
upd = VaspRelaxUpdater().apply_preset(structure, 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 1}
                                     )
upd.set_resources(tot_num_mpiprocs=32, num_machines=1)
upd.set_options(max_wallclock_seconds=3600, queue_name='xhhctdnormal')
upd.set_label(f'{structure.get_pymatgen().composition.reduced_formula} RELAX')
upd.set_incar(symprec=1e-3)
upd.builder

running = upd.submit()
# Note that this is actually the 8 atom conventional cell
workpath.add_node(running, f'{elem}_bulk', True)

Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

/home/bonan/miniconda3/envs/aiida/lib/python3.12/site-packages/aiida_user_addons/tools/pymatgen.py:57: AiidaDeprecationWarning: `StructureData.set_extra` is deprecated, use `StructureData.base.extras.set` instead. (this will be removed in v3)
  strucd.set_extra("mp_id", mp_id)
/home/bonan/miniconda3/envs/aiida/lib/python3.12/site-packages/aiida_user_addons/tools/pymatgen.py:59: AiidaDeprecationWarning: `StructureData.set_extra` is deprecated, use `StructureData.base.extras.set` instead. (this will be removed in v3)
  strucd.set_extra("mp_magmom", magmom)


Au


## Compute for bulk relaxation

The bulk structure needs to be generated or loaded, here we use TlCuSe2 as a template structure to generate
other Chalcopyrite structures. 

The structure is then fully relaxed in order as it is needed as the reference bulk structure for creating defect supercells and the prestine supercell structure.

In [24]:
structure = load_mp_struct('mp-14090')

ps = structure.get_pymatgen()
ps['Tl'] = 'In'  # A site - CHANGE manually here
ps['Cu'] = 'Au'  # B site - CHANGE manually here
ps['Se'] = 'Se'  # C site - CHANGE manually here
structure = orm.StructureData(pymatgen=ps)
print(structure.get_formula())
upd = VaspRelaxUpdater().apply_preset(structure, 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 1, 'symprec': 1e-3},
                                     )
upd.set_resources(tot_num_mpiprocs=32, num_machines=1)
upd.set_options(max_wallclock_seconds=3600, queue_name='xhhctdnormal')
upd.set_label(f'{structure.get_pymatgen().composition.reduced_formula} RELAX')
upd.set_relax_settings(algo='rd')
upd.builder


running = upd.submit()
# Note that this is actually the 8 atom conventional cell
workpath[f'{FORMULA.lower()}_bulk'] = running

Au2In2Se4


/home/bonan/aiida_env/aiida-2.0/aiida-grouppathx/aiida_grouppathx/pathx.py:495: UserWarning: Unsetting existing node: uuid: 95d8b01c-437f-4012-bbb7-bf84677ac56b (pk: 663384) (aiida.workflows:vasp.relax)'s alias
  warnings.warn(f"Unsetting existing node: {target_path.node}'s alias")


In [25]:
workpath.show_tree(decorate_with_exit_status)

workflows
├── Au_bulk [0]
├── In_bulk [0]
├── Se_bulk [0]
├── inause2_222_V_Au [500]
├── inause2_222_V_In [500]
├── inause2_222_V_Se [500]
├── inause2_222_supercell [500]
└── inause2_bulk [waiting]



The cell below can be used to inspect the structure.
NOTE: `ase-weas-widget` needs to be installed in order to use `weas-widget` to view the structure via jupyter.

In [13]:
from ase.visualize import view


view(workpath[f'{FORMULA.lower()}_bulk'].node.inputs.structure.get_ase(), viewer='weas')

WeasWidget(children=(BaseWidget(atoms={'species': {'In': 'In', 'Au': 'Au', 'Se': 'Se'}, 'cell': [5.37034214, 0…

## Proceed with defect calculation

The bulk structure is need to generate supercell and defect supercells

In [27]:

ref_node = workpath[f'{FORMULA.lower()}_bulk'].node
# What until bulk calculation is finished
while not workpath[f'{FORMULA.lower()}_bulk'].node.is_finished_ok:
    print(f"Waiting relaxation {ref_node} to finish ") 
    time.sleep(300)

# Takae the relaxed bulk structure as reference structure
ref_structure = workpath[f'{FORMULA.lower()}_bulk'].node.outputs.relax.structure
# Elements to make vacancy of
elems = set(ref_structure.get_ase().symbols) # Elements to make vacancy

for elem in elems:
    # Find the first occurance of the element
    # This assumes all atoms of the same element are equivalent by symmetry, which may not be the case
    # A better way is to spglib to find unique sites of each element and calculate for them all
    i_elem = ref_structure.get_ase().get_chemical_symbols().index(elem) 
    #  This generate a vacancy cell
    vac_cell = make_vac(ref_structure, [i_elem] , [2,2,2])
    print('Defect cell formula', vac_cell.get_formula())
    upd = VaspRelaxUpdater().apply_preset(vac_cell, 
                                          code='vasp-6.3.2@sugon-xh-v2',
                                          overrides={'ispin': 1,  # I use ISPIN=1 for simplicty, also it may OVERESTIMATE the formation energy
                                                     'ncore':8, 'kpar':4, 'lorbit': None, 'symprec':1e-3}
                                         )
    # Consider lowering the number of core/machines for high-throughput calculations
    upd.set_resources(tot_num_mpiprocs=128, num_machines=2) 
    upd.set_options(max_wallclock_seconds=3600 * 12, queue_name='xhhctdnormal')
    upd.set_label(f'{FORMULA} 222 V_{elem} RELAX')  # Set the label
    upd.set_relax_settings(volume=False, shape=False)  # Relax only ionic positions
    upd.builder
    running = upd.submit()
    
    workpath.add_node(running, f'{FORMULA.lower()}_222_V_{elem}')


# Submit reference bulk supercell calculation

ref_222 = make_supercell(workpath[f'{FORMULA.lower()}_bulk'].node.outputs.relax.structure,[2,2,2])['structure']
upd = VaspRelaxUpdater().apply_preset(ref_222, 
                                      code='vasp-6.3.2@sugon-xh-v2',
                                      overrides={'ispin': 1,
                                                 'ncore':8, 'kpar':4, 'lorbit': None,}
                                     )
# Consider lowering the number of core/machines for high-throughput calculations
upd.set_resources(tot_num_mpiprocs=128, num_machines=2)
upd.set_options(max_wallclock_seconds=3600 * 12, queue_name='xhhctdnormal')
upd.set_label(f'{FORMULA} 222 SUPERCELL')
upd.set_relax_settings(volume=False, shape=False)  # Relax only ionic positions
upd.builder

running = upd.submit()

workpath.add_node(running, f'{FORMULA.lower()}_222_supercell')

Waiting relaxation uuid: 7100dfa5-e2f4-4825-87b0-689489a971c2 (pk: 663523) (aiida.workflows:vasp.relax) to finish 
Waiting relaxation uuid: 7100dfa5-e2f4-4825-87b0-689489a971c2 (pk: 663523) (aiida.workflows:vasp.relax) to finish 


06/01/2025 05:44:14 PM <2538389> aiida.engine.processes.functions: [INFO] Executing process function, current stack status: 25 frames of 3000
06/01/2025 05:44:14 PM <2538389> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<663546>: Broadcasting state change: state_changed.created.running
06/01/2025 05:44:14 PM <2538389> aiida.orm.nodes.process.calculation.calcfunction.CalcFunctionNode: [INFO] Process<663546>: Broadcasting state change: state_changed.running.finished


Defect cell formula Au16In15Se32


PathIsNotVirtualError: This path corresponds a group or a node: inause2-defect/workflows/inause2_222_V_In

In [18]:
workpath.show_tree()

workflows
├── Au_bulk *
├── In_bulk *
├── Se_bulk *
├── inause2_222_V_Au *
├── inause2_222_V_In *
├── inause2_222_V_Se *
├── inause2_222_supercell *
└── inause2_bulk *



## Compute the formation energy

In [19]:
def read_energy(path):
    """Return the total energy"""
    return path.get_node().outputs.misc['total_energies']['energy_extrapolated']
    
def read_energy_per_atom(path):
    """Return the energy per atom"""
    node = path.get_node()
    eng = node.outputs.misc['total_energies']['energy_extrapolated']
    return eng / len(node.inputs.structure.sites)

def show_formation_energy(supercell, v_hg, elemental):
    """Show and return the vacancy formation energy given the supercell, vacancy-containing cell
    and the elemental reference calculation.
    The vacancy formation energy returns assumes the X-rich limit where the system is in
    equilibirum with the elemental phase.
    """
    elem = workpath[elemental].node.outputs.relax.structure.get_formula()
    evac = read_energy(workpath[v_hg])
    print(f'Vacancy bearing cell: {evac:.5f} eV')
    ebulk = read_energy(workpath[supercell])
    print(f'Bulk cell: {ebulk: .5f} eV')
    e_hg = read_energy_per_atom(workpath[elemental])
    print(f'Energy per {elem} atom: {e_hg: .5f} eV')
    e_vac = evac + e_hg - ebulk
    print(f'Vacancy formation energy: {e_vac:.5f} eV')
    return e_vac

In [20]:
forms = {}
for elem in elems:
    forms[elem] = show_formation_energy(f'{FORMULA.lower()}_222_supercell', 
                                        f'{FORMULA.lower()}_222_V_{elem}', f'{elem}_bulk')

NotExistentAttributeError: Node<663431> does not have an output with link label 'misc'

In [21]:
forms

{}

## Check energy and chemical formula using show_tree

This can be used to manually validate the vacancy formation energy calculated

In [22]:
def form(path):
    if path.is_node:
        return path.node.inputs.structure.get_formula()
def energy(path):
    if path.is_node:
        if not path.node.is_finished_ok:
            return
        return '{:.4f} eV'.format(path.node.outputs.misc['total_energies']['energy_extrapolated'])

In [23]:
workpath.show_tree(form, energy)

workflows
├── Au_bulk Au | -3.9481 eV
├── In_bulk In | -2.9867 eV
├── Se_bulk Se3 | -11.5568 eV
├── inause2_222_V_Au Au15In16Se32
├── inause2_222_V_In Au16In15Se32
├── inause2_222_V_Se Au16In16Se31
├── inause2_222_supercell Au16In16Se32
└── inause2_bulk Au2In2Se4

